In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy

In [ ]:
# data columns:
#     name
#     milogP (from https://www.molinspiration.com/cgi/properties)
#     current decay time constant (s) in washout experiments, AKA washout time constant
#     R group name
#     R group logP (from pubchem)  
#     comments to the right are the sources cited by pubchem
data = [["GLPG1837", 1.76, 17.4,   "pyrazole",            0.26], #HANSCH,C ET AL. (1995)
        ["JM97",     2.59, 56.5,   "phenylethyl alcohol", 1.36], #Hansch, C. and A. Leo. The Log P Database. Claremont, CA: Pomona College, 1987.
        ["JM112",    3.74, 243.02, "1-heptanol",          2.62], #Sangster J; Log KOW Databank. Sangster Res Lab. (1994)
        ["JM96",     5.76, 2166.5, "1-undecanol",         4.72]  #Abraham MH et al; J Pharm Sci 83: 1085-100 (1994)
       ]

labels_wm = [d[0] for d in data]
labels_r = [d[3] for d in data]

washout_time_constants = [d[2] for d in data]
washout_rates = [1/d[2] for d in data]

#predicted membrane-water partitioning time constants and rates using eq. 34.3 with m=1 chosen arbitrarily
r_group_logp_time_constants  = [10**(d[4]/2) for d in data]
r_group_logp_rates = [10**(-d[4]/2) for d in data]

whole_molecule_milogp_time_constants  = [10**(d[1]/2) for d in data]
whole_molecule_milogp_rates  = [10**(-d[1]/2) for d in data]

In [ ]:
def plot_rate_relations(x_rates, y_rates, xlabel, ylabel, pointlabels, labelspacing, xlim, ylim, plottitle, include_nonzero_intercept_fit = False):

    #plot data
    fig, ax = plt.subplots()
    plt.scatter(x_rates, y_rates, color="black")

    #plot labels
    for i, txt in enumerate(pointlabels):
        ax.annotate(txt, (x_rates[i]+labelspacing, y_rates[i]))
    
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    #plot range
    if xlim != ():
        plt.xlim(xlim)
    if ylim != ():
        plt.ylim(ylim)

    #linear fit with intercepts fixed at 0,0 to match the functional form of eq. 34
    #ignore the warning thrown here, which originates somewhere within linalg.lstsq and is not fixable by adjusting the parameters here
    slope_0int, residuals, rank, s = np.linalg.lstsq(np.array(x_rates)[:,np.newaxis], y_rates)
    
    #calculate the r^2 value from the residuals
    total_sum_of_squares = sum([(wr-np.mean(y_rates))**2 for wr in y_rates])
    r_squared_0int = 1-residuals[0]/total_sum_of_squares
    print(f"r-squared = {r_squared_0int}")
    
    #plot best fit line, extending to the intercept at 0,0
    predicted_values_0int = slope_0int*np.array([0]+x_rates)
    plt.plot([0]+x_rates, predicted_values_0int)
    
    if include_nonzero_intercept_fit:
        #linear fit including a nonzero intercept
        # this fit finds that the optimal intercept is nearly 0 as predicted by eq. 34.
        
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(x_rates, y_rates)
        print(slope, intercept, r_value, p_value, std_err)
        
        predicted_values = slope*np.array(x_rates) + intercept*np.ones(len(x_rates))
        plt.plot(x_rates, predicted_values, linestyle="dashed")
    
        print(intercept/np.mean(x_rates))
    
    if plottitle != "":
        plt.savefig(plottitle, dpi=600)

In [ ]:
x_rates = r_group_logp_rates
y_rates = washout_rates
xlabel = "Membrane unbinding rate predicted from R group logP\n(arb. units; inverse time)"
ylabel = "washout rate (1/s)"
pointlabels = labels_r
labelspacing = 0.01
plottitle = "r_logp_rate_vs_washout_rate"
xlim = (0,0.9)
ylim = (0,0.06)
plot_rate_relations(x_rates, y_rates, xlabel, ylabel, pointlabels, labelspacing, xlim, ylim, plottitle)

In [ ]:
x_rates = r_group_logp_time_constants
y_rates = washout_time_constants
xlabel = "Membrane unbinding time constant predicted\nfrom R group logP (arb. units; time)"
ylabel = "washout time constant (s)"
pointlabels = labels_r
label_spacing = 5
plottitle = "r_logp_time_constant_vs_washout_time_constant"
xlim = (0,300)
ylim = (0,2250)
plot_rate_relations(x_rates, y_rates, xlabel, ylabel, pointlabels, label_spacing, xlim, ylim, plottitle)

In [ ]:
x_rates = whole_molecule_milogp_rates
y_rates = washout_rates
xlabel = "Membrane unbinding rate predicted from whole molecule milogP\n(arb. units; inverse time)"
ylabel = "washout rate (1/s)"
pointlabels = labels_wm
labelspacing = 0.002
plottitle = "wm_milogp_rate_vs_washout_rate"
xlim = (0,0.16)
ylim = (0,0.06)
plot_rate_relations(x_rates, y_rates, xlabel, ylabel, pointlabels, labelspacing, xlim, ylim, plottitle)

In [ ]:
x_rates = whole_molecule_milogp_time_constants
y_rates = washout_time_constants
xlabel = "Membrane unbinding time constant predicted\nfrom whole molecule milogP (arb. units; time)"
ylabel = "washout time constant (s)"
pointlabels = labels_wm
label_spacing = 20
plottitle = "wm_milogp_time_constant_vs_washout_time_constant"
xlim = (0,900)
ylim = (0,2250)
plot_rate_relations(x_rates, y_rates, xlabel, ylabel, pointlabels, label_spacing, xlim, ylim, plottitle)